# Document Question Answering using Retrieval-Augmented Generation (RAG)

**Project Description:**
This notebook implements a Retrieval-Augmented Generation (RAG) system that answers questions from a custom PDF document. Instead of relying purely on a language model's internal (and possibly outdated or incomplete) knowledge, the system retrieves relevant text chunks from the document and grounds its generated answers in that retrieved context. This makes the system suitable for question-answering over private or domain-specific data such as notes, resumes, research papers, or books.

## Objectives

- Understand the concept of Retrieval-Augmented Generation (RAG)
- Build a pipeline combining retrieval and generation
- Answer questions using custom PDF documents
- Learn how embeddings and vector databases work
- Improve factual grounding of LLM answers using retrieved context

## 1. Install Required Libraries

We install only the libraries required for this project:
- `langchain` and `langchain-community` — orchestration of the RAG pipeline
- `langchain-huggingface` — HuggingFace integration for LangChain
- `sentence-transformers` — free embedding model
- `faiss-cpu` — vector database for similarity search
- `pypdf` — PDF text extraction
- `transformers`, `accelerate`, `torch` — running the free local language model

> Run this cell once. In Google Colab, a restart may occasionally be required after installation — if imports fail in the next cell, use *Runtime > Restart Runtime* and re-run.

In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters \
    sentence-transformers faiss-cpu pypdf transformers accelerate torch

## 2. Import Libraries

All imports are organized by purpose: document loading, chunking, embeddings, vector storage, and language model generation.

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

# Force transformers to use PyTorch only (avoids TensorFlow/Keras 3
# compatibility errors on systems where TensorFlow is also installed)
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# Document loading
from langchain_community.document_loaders import PyPDFLoader

# Text chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_community.vectorstores import FAISS

# Language model
from transformers import pipeline

print("All libraries imported successfully.")

All libraries imported successfully.


## 3. Upload Your PDF Document

Since RAG is meant to work over your own custom data, select a PDF as the knowledge source. The cell below works in both environments:

- **Google Colab** — opens the built-in upload picker
- **Local Jupyter / VS Code** — opens a native file-selection dialog (or, if that's unavailable, prompts you to type the file path)

Either way, the selected file's path is stored automatically in `PDF_PATH`.

In [3]:
try:
    # Running in Google Colab
    from google.colab import files

    uploaded = files.upload()  # opens a file picker to select your PDF
    if len(uploaded) == 0:
        raise ValueError("No file was uploaded. Please re-run this cell and select a PDF.")
    PDF_PATH = list(uploaded.keys())[0]
    print(f"File uploaded successfully: {PDF_PATH}")

except ImportError:
    # Running locally (e.g. VS Code, JupyterLab, plain Jupyter Notebook)
    try:
        # Try a native file-picker dialog
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()          # hide the empty root window
        root.attributes("-topmost", True)
        PDF_PATH = filedialog.askopenfilename(
            title="Select a PDF document",
            filetypes=[("PDF files", "*.pdf")]
        )
        root.destroy()

        if not PDF_PATH:
            raise ValueError("No file was selected.")

    except Exception:
        # Final fallback: type the path manually
        PDF_PATH = input("Enter the full path to your PDF file: ").strip().strip('"')

    print(f"PDF selected: {PDF_PATH}")

PDF selected: C:/Users/devan/Downloads/Research_paper_PBL .pdf


## 4. Load PDF

Specify the path to the PDF document you want to use as the knowledge source. The notebook makes no assumptions about the document's contents — any PDF (notes, resume, research paper, book, etc.) can be used.

In [4]:
def load_pdf(pdf_path: str):
    """
    Load a PDF file and return a list of LangChain Document objects,
    one per page. Handles missing files and empty PDFs gracefully.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    if len(documents) == 0:
        raise ValueError("The PDF appears to be empty or could not be read.")

    return documents


documents = load_pdf(PDF_PATH)

total_characters = sum(len(doc.page_content) for doc in documents)

print(f"PDF loaded successfully: {PDF_PATH}")
print(f"Number of pages: {len(documents)}")
print(f"Total characters extracted: {total_characters:,}")

PDF loaded successfully: C:/Users/devan/Downloads/Research_paper_PBL .pdf
Number of pages: 5
Total characters extracted: 19,750


## 5. Text Chunking

Long documents are split into smaller overlapping chunks. This improves retrieval accuracy, since embedding a very long page tends to blur together multiple topics, while overlap ensures we don't lose context at chunk boundaries.

- `chunk_size = 500` characters
- `chunk_overlap = 100` characters

In [5]:
def split_documents(documents, chunk_size=500, chunk_overlap=100):
    """
    Split a list of Documents into smaller overlapping text chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)

    if len(chunks) == 0:
        raise ValueError("No text chunks were created. The PDF may contain no extractable text.")

    return chunks


CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

chunks = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

avg_chunk_length = sum(len(c.page_content) for c in chunks) / len(chunks)

print(f"Number of chunks created: {len(chunks)}")
print(f"Chunk size (target): {CHUNK_SIZE} characters")
print(f"Chunk overlap: {CHUNK_OVERLAP} characters")
print(f"Average actual chunk length: {avg_chunk_length:.0f} characters")
print("\nExample chunk:\n")
print(chunks[0].page_content)

Number of chunks created: 50
Chunk size (target): 500 characters
Chunk overlap: 100 characters
Average actual chunk length: 456 characters

Example chunk:

A Machine Learning-Based Multi-Class Text
Classification System for Intelligent Message
Filtering
Devansh Mehrotra, Neha V Sharma
Department of Data Science Engineering
Manipal University Jaipur
Rajasthan, India
nehav.sharma@jaipur.manipal.edu
Abstract—With the ever-increasing volume of short messages
being sent through platforms like SMS and social media, in-
telligent filtering methods are essential to avoid information
overload. Existing messaging classifiers mostly use a binary


## 6. Create Embeddings

Each text chunk is converted into a dense vector representation that captures its semantic meaning. We use a free, lightweight HuggingFace sentence-embedding model: `sentence-transformers/all-MiniLM-L6-v2`.

> **What does the embedding dimension mean?** Each chunk is represented as a point in a high-dimensional space (384 dimensions for `all-MiniLM-L6-v2`). Chunks with similar meaning end up close together in this space, which is exactly what lets us find relevant text via similarity search rather than exact keyword matching.

In [6]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

# Quick sanity check: embed the first chunk
sample_vector = embedding_model.embed_query(chunks[0].page_content)

print(f"Embedding model loaded: {EMBEDDING_MODEL_NAME}")
print(f"Embedding vector dimension: {len(sample_vector)}")

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding vector dimension: 384


## 7. Create Vector Database

We store all chunk embeddings in a FAISS vector database. FAISS enables fast similarity search so that, given a query embedding, we can quickly find the most semantically similar chunks.

In [7]:
vector_store = FAISS.from_documents(chunks, embedding_model)

print("Vector database created successfully.")
print(f"Total vectors stored: {vector_store.index.ntotal}")

print(f"Distance metric: L2 (Euclidean) — lower scores indicate higher similarity.")

Vector database created successfully.
Total vectors stored: 50
Distance metric: L2 (Euclidean) — lower scores indicate higher similarity.


## 8. Create Retriever

Note: the retriever object below demonstrates LangChain's standard retrieval interface, as required for this assignment. However, it does not expose similarity scores. Since Section 11 (Question Answering) displays retrieval scores alongside each chunk, the actual answer-generation pipeline calls vector_store.similarity_search_with_score() directly instead of using this retriever object. Both approaches query the same underlying FAISS index and return identical chunks — the only difference is whether scores are included.

The retriever performs similarity search over the FAISS vector database and returns the Top-3 most relevant chunks for a given query.

> **Note on similarity scores:** The `retriever` object above is used for the standard retrieval interface, but it does not expose similarity scores by default. To display scores alongside retrieved chunks (Section 11), we call `vector_store.similarity_search_with_score()` directly, which returns each chunk together with its distance score.

In [8]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

print("Retriever configured: returns top-3 most relevant chunks per query.")

Retriever configured: returns top-3 most relevant chunks per query.


## 9. Load a Language Model

We use a free, locally-runnable HuggingFace text-generation model: `google/flan-t5-base`. This model is small enough to run in Google Colab without a paid API key or GPU (though a GPU will speed things up).

In [9]:
LLM_MODEL_NAME = "google/flan-t5-base"

# flan-t5-base was pretrained with a 512-token encoder input limit.
# We load the tokenizer explicitly so we can measure and control how many
# tokens of retrieved context we actually send in, rather than letting the
# pipeline silently truncate (which can cut context off mid-sentence and
# degrade answer quality).
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

generator = pipeline(
    "text2text-generation",
    model=LLM_MODEL_NAME,
    tokenizer=tokenizer
)

# do_sample=False makes decoding greedy/deterministic: the same question will
# always produce the same answer, which is preferable for document QA (we want
# consistent, reproducible answers grounded in the retrieved context, not
# creative variation).

MODEL_MAX_INPUT_TOKENS = 512   # flan-t5-base encoder limit
MAX_NEW_TOKENS = 300           # length budget for the generated answer only

print(f"Language model loaded: {LLM_MODEL_NAME}")
print(f"Max input tokens: {MODEL_MAX_INPUT_TOKENS}, Max new tokens: {MAX_NEW_TOKENS}")

Device set to use cpu


Language model loaded: google/flan-t5-base
Max input tokens: 512, Max new tokens: 300


## 10. Build the RAG Pipeline

The full pipeline works as follows:

```
User Question
      ↓
Convert question into embedding
      ↓
Retrieve relevant chunks from FAISS
      ↓
Combine retrieved context
      ↓
Create prompt
      ↓
Send prompt to language model
      ↓
Generate grounded answer
```

The function below implements this end-to-end.

In [10]:
import torch


def truncate_context_to_token_budget(context: str, question: str, tokenizer,
                                      model_max_tokens: int = MODEL_MAX_INPUT_TOKENS,
                                      reserved_for_output: int = MAX_NEW_TOKENS) -> str:
    """
    Ensure the retrieved context fits within the model's actual input
    token limit, leaving room for the instructions/question text.
    Truncates by tokens (not characters) so we don't silently cut off
    mid-sentence in a way the model can't recover from.
    """
    instruction_template = (
        "Answer the question using only the context below. "
        "If the answer is not contained in the context, say "
        "\"The document does not provide this information.\"\n\n"
        "Context:\n\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    instruction_tokens = len(tokenizer.encode(instruction_template))

    # Budget left over for the context itself
    available_tokens = model_max_tokens - instruction_tokens
    available_tokens = max(available_tokens, 0)

    context_tokens = tokenizer.encode(context, truncation=True, max_length=available_tokens)
    truncated_context = tokenizer.decode(context_tokens, skip_special_tokens=True)

    return truncated_context


def build_prompt(question: str, context: str) -> str:
    """
    Construct a prompt that instructs the model to answer strictly
    using the provided context.
    """
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not contained in the context, say "
        "\"The document does not provide this information.\"\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt


def retrieve_relevant_chunks(question: str, vector_store, k: int = 3):
    """
    Retrieve the top-k most relevant chunks along with their similarity
    scores and page metadata. Using the vector store directly (rather than
    the retriever wrapper) gives us access to the distance scores.
    """
    results = vector_store.similarity_search_with_score(question, k=k)
    return results  # list of (Document, score) tuples


def answer_question(question: str, vector_store, generator, tokenizer, k: int = 3):
    """
    Run the full RAG pipeline for a single question:
    retrieve relevant chunks (with scores + page numbers), build a
    grounded prompt that fits within the model's token limit, and
    generate a deterministic answer.
    """
    retrieved = retrieve_relevant_chunks(question, vector_store, k=k)
    raw_context = "\n\n".join(doc.page_content for doc, score in retrieved)

    # Enforce the model's token budget so context isn't silently
    # truncated mid-sentence by the pipeline itself
    context = truncate_context_to_token_budget(raw_context, question, tokenizer)

    # Create the grounded prompt
    prompt = build_prompt(question, context)

    # Generate the answer. torch.no_grad() disables gradient tracking during
    # inference (the HF pipeline already does this internally, but it is made
    # explicit here for clarity). do_sample=False and truncation=True keep
    # answers deterministic and safely within the model's input limit.
    with torch.no_grad():
        result = generator(
            prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            truncation=True
        )[0]["generated_text"]

    return result, retrieved


def print_retrieved_context(retrieved):
    """
    Neatly display retrieved chunks with their source page number and
    similarity score.
    """
    for i, (doc, score) in enumerate(retrieved, start=1):
        page_number = doc.metadata.get("page", None)
        page_display = f"Page {page_number + 1}" if page_number is not None else "Page unknown"

        print("=" * 60)
        print(f"Chunk {i} ({page_display}) | Distance score: {score:.4f} (lower = more similar)")
        print("-" * 60)
        print(doc.page_content)
    print("=" * 60)


print("RAG pipeline functions ready.")

RAG pipeline functions ready.


## 11. Question Answering

Run the cell below and type a question when prompted. The system will display the question, the retrieved context, and the generated answer.

In [16]:
question = input("Enter your question: ")

answer, retrieved = answer_question(question, vector_store, generator, tokenizer)

print("\n" + "#" * 60)
print("USER QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print_retrieved_context(retrieved)

print("\nGENERATED ANSWER:")
print(answer)
print("#" * 60)


############################################################
USER QUESTION:
which is the main model used in this

RETRIEVED CONTEXT:
Chunk 1 (Page 3) | Distance score: 1.3668 (lower = more similar)
------------------------------------------------------------
numpyused for data manipulation andscikit-learnfor
machine learning. Model and vectorizer objects are saved sep-
arately usingjoblibin order to be deployed independently
of training code.
Frontend rendering is performed usingFlaskmicroframe-
work [?]. There are two routes implemented:
1) Single Message Prediction (/predict):POST route
that takes a raw string from HTML form. Raw string is cleaned
withclean_text()method, vectorized and predicted. With
Chunk 2 (Page 2) | Distance score: 1.5008 (lower = more similar)
------------------------------------------------------------
multi-classification task, the One-Vs-Rest (OvR) method was
applied. In OvR, one binary classifier per classcis constructed
to classify the current class versus

## 12. Example Questions

A few example questions to demonstrate the system. These are generic and should work on any uploaded PDF.

In [12]:
example_questions = [
    "What is the main objective of this document?",
    "Summarize the document.",
    "What are the key concepts discussed?"
]

for q in example_questions:
    answer, retrieved = answer_question(q, vector_store, generator, tokenizer)

    print("\n" + "#" * 60)
    print(f"USER QUESTION: {q}")

    print("\nRETRIEVED CONTEXT:")
    print_retrieved_context(retrieved)

    print("\nGENERATED ANSWER:")
    print(answer)
    print("#" * 60)


############################################################
USER QUESTION: What is the main objective of this document?

RETRIEVED CONTEXT:
Chunk 1 (Page 1) | Distance score: 1.4471 (lower = more similar)
------------------------------------------------------------
involving data preprocessing and feature extraction. Section IV
describes the architecture of the proposed system. Section V
discusses the evaluation process and findings, while Section
VI highlights the limitations of the study and future work.
II. RELATEDWORK
A large body of literature exists regarding automated text
categorization, from early rule-based systems to more sophis-
ticated models in machine learning and deep learning domains
[3].
Chunk 2 (Page 1) | Distance score: 1.5368 (lower = more similar)
------------------------------------------------------------
and TF-IDF [4] have been the cornerstone techniques for
mapping unstructured text data to numerical vectors. Text
classification is an intrinsically high dim

## 13. Workflow Diagram

```
PDF
 ↓
Text Extraction
 ↓
Chunking
 ↓
Embeddings
 ↓
FAISS Vector Store
 ↓
User Query
 ↓
Retriever
 ↓
LLM
 ↓
Answer
```

## 14. Conclusion

This project demonstrated how to build a Retrieval-Augmented Generation (RAG) system for question answering over custom documents. Key takeaways:

- **Retrieval**: Finding the most semantically relevant chunks of text is central to grounding an LLM's answer in real data, rather than relying on its parametric knowledge alone.
- **Embeddings**: Dense vector representations allow us to compare the *meaning* of text, not just keyword overlap, enabling semantic search.
- **Vector Databases**: FAISS provides an efficient way to store and search over large numbers of embeddings.
- **Language Models**: A general-purpose LLM (here, `flan-t5-base`) can produce accurate, grounded answers when given the right context — even without being fine-tuned on the specific document.
- **Retrieval-Augmented Generation**: Combining retrieval and generation improves factual accuracy and allows question answering over private or domain-specific data that the LLM was never trained on.

RAG systems like this one form the foundation of many real-world applications, including chatbots, knowledge assistants, enterprise search tools, and AI-powered documentation systems.